# CARE++ — Prediction Component


## 0. Setup and reproducibility

In [ ]:
import os
import copy
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


def set_seed(seed):
    """Reset every RNG we use so a run is reproducible given a seed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Main seed used for the "official" single-run pipeline below.
# The 5-seed robustness check (Section 10) additionally re-runs everything
# under SEEDS = [42, 7, 13, 123, 2024].
SEED = 42
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


## 1. Load data

In [ ]:
TRAIN_PATH  = "/content/train_df_gps.parquet"
VAL_PATH    = "/content/val_df_gps.parquet"
TEST_PATH   = "/content/test_df_gps.parquet"
DEVICE_PATH = "/content/safecast_device_summary.csv"

train_df  = pd.read_parquet(TRAIN_PATH)
val_df    = pd.read_parquet(VAL_PATH)
test_df   = pd.read_parquet(TEST_PATH)
device_df = pd.read_csv(DEVICE_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Device summary shape:", device_df.shape)


Train shape: (1730820, 37)
Validation shape: (1070390, 37)
Test shape: (355967, 37)
Device summary shape: (24, 14)


### GPS column check

Your data has two candidate location sources: `lat`/`lon` (from the raw
Safecast join, ~2-5% match rate before the ASOF fix) and the pre-existing
`latitude`/`longitude`/`location_name` columns that were already in the
original prepared data. This reports fill rate for both so you can see
which one is actually usable before anything downstream relies on either.

In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cols_present = [c for c in ["lat", "lon", "latitude", "longitude", "location_name"] if c in df.columns]
    print(f"\n{name} ({len(df):,} rows) — columns present: {cols_present}")
    for c in cols_present:
        filled = df[c].notna().sum()
        print(f"  {c:15s}: {filled:>10,} non-null ({filled/len(df)*100:5.2f}%)")



train (1,730,820 rows) — columns present: ['lat', 'lon', 'latitude', 'longitude', 'location_name']
  lat            :     33,116 non-null ( 1.91%)
  lon            :     33,116 non-null ( 1.91%)
  latitude       :  1,730,820 non-null (100.00%)
  longitude      :  1,730,820 non-null (100.00%)
  location_name  :  1,320,098 non-null (76.27%)

val (1,070,390 rows) — columns present: ['lat', 'lon', 'latitude', 'longitude', 'location_name']
  lat            :     53,215 non-null ( 4.97%)
  lon            :     53,215 non-null ( 4.97%)
  latitude       :  1,070,390 non-null (100.00%)
  longitude      :  1,070,390 non-null (100.00%)
  location_name  :    678,871 non-null (63.42%)

test (355,967 rows) — columns present: ['lat', 'lon', 'latitude', 'longitude', 'location_name']
  lat            :      6,250 non-null ( 1.76%)
  lon            :      6,250 non-null ( 1.76%)
  latitude       :    355,967 non-null (100.00%)
  longitude      :    355,967 non-null (100.00%)
  location_name  :    184,5

## 2. Feature and target configuration

In [ ]:
feature_cols = [
    "cpm_mean",
    "cpm_median",
    "cpm_std",
    "log_cpm_mean",
    "previous_cpm",
    "cpm_change",
    "cpm_change_pct",
    "past_cpm_mean_5",
    "past_cpm_std_5",
    "past_cpm_mean_15",
    "past_cpm_std_15",
    "past_cpm_mean_30",
    "past_cpm_std_30",
    "zscore_5",
    "zscore_15",
    "zscore_30",
    "hour",
    "device_quality_score",
]

target_col = "log_cpm_mean"

print("Number of input features:", len(feature_cols))
print("Target:", target_col)

# Note: "log_cpm_mean" appears both as an input feature and as the target
# column. This is NOT leakage: build_sequences() below only uses feature
# values from the window *before* the predicted timestep, never the value
# being predicted. It just means the model gets its own recent log-CPM
# history as an autoregressive input, which is standard practice.


Number of input features: 18
Target: log_cpm_mean


## 3. Feature scaling (fixes the CNN blow-up)

The root cause of the CNN's exploding validation loss: `feature_cols` mixes raw
CPM values (can be in the hundreds), an unbounded raw `hour` (0–23), an
unknown-scale `device_quality_score`, and already-standardized z-scores
(roughly -3 to 3) — all fed into the same network with no normalization.

Fit a `StandardScaler` **on `train_df` only**, then reuse that exact fitted
scaler everywhere else (validation, test, and later every federated client).
Fitting on train only avoids leaking validation/test statistics into
training.

In [ ]:
scaler = StandardScaler()
scaler.fit(train_df[feature_cols].values)

print("Fitted StandardScaler on", len(feature_cols), "columns using train_df only.")
print("Per-feature mean (train):", np.round(scaler.mean_, 3))
print("Per-feature scale (train):", np.round(scaler.scale_, 3))


Fitted StandardScaler on 18 columns using train_df only.
Per-feature mean (train): [ 1.21519e+02  1.20496e+02  1.85120e+01  3.11200e+00  1.21520e+02
 -2.00000e-03  4.00000e-02  1.21521e+02  1.41080e+01  1.21525e+02
  1.96070e+01  1.21537e+02  2.55560e+01  2.99000e-01  2.37000e-01
  2.17000e-01  1.14990e+01  9.44950e+01]
Per-feature scale (train): [757.439 755.291 132.773   0.822 757.443 133.546   2.198 750.122 116.646
 740.777 162.46  730.667 201.587  26.727  21.574  20.155   6.928   4.107]


## 4. Sequence-building utilities (scaler-aware, with baseline predictions)

In [ ]:
def build_sequences(
    df,
    feature_cols,
    target_col,
    window_size=10,
    scaler=None,
    ma_window=5,
):
    """Build sliding-window sequences.

    Returns X, y, and two baseline predictions (persistence and moving
    average) computed purely from the RAW (unscaled) target history, so they
    can be evaluated with the exact same metric code as the trained models.

    `scaler`, if given, is applied only to the array used to build X — the
    target column used to build y is always taken from the raw, unscaled
    values, even when target_col also happens to be one of feature_cols.
    """
    df = df.sort_values("timestamp").reset_index(drop=True)

    raw_features = df[feature_cols].values.astype(np.float32)
    if scaler is not None:
        features = scaler.transform(raw_features).astype(np.float32)
    else:
        features = raw_features

    targets = df[target_col].values.astype(np.float32)

    X_list, y_list = [], []
    persistence_list, ma_list = [], []

    for i in range(len(df) - window_size):
        X_list.append(features[i:i + window_size])
        y_list.append(targets[i + window_size])

        # Persistence baseline: "tomorrow looks like today" —
        # the last observed value before the point being predicted.
        persistence_list.append(targets[i + window_size - 1])

        # Moving-average baseline over the last `ma_window` raw observations
        # immediately preceding the predicted point.
        ma_start = max(0, i + window_size - ma_window)
        ma_list.append(np.mean(targets[ma_start:i + window_size]))

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    persistence_pred = np.array(persistence_list, dtype=np.float32)
    ma_pred = np.array(ma_list, dtype=np.float32)

    return X, y, persistence_pred, ma_pred


def partition_by_device(df, device_col="device_id"):
    clients = {}
    for device_id, sub_df in df.groupby(device_col):
        clients[device_id] = (
            sub_df.sort_values("timestamp").reset_index(drop=True)
        )
    return clients


def build_device_sequences(
    df,
    feature_cols,
    target_col,
    window_size=10,
    scaler=None,
    ma_window=5,
):
    device_data = partition_by_device(df, device_col="device_id")

    sequences = {}

    for device_id, device_df in device_data.items():
        required_cols = feature_cols + [target_col]
        device_df = device_df.dropna(subset=required_cols)

        if len(device_df) <= window_size:
            continue

        X, y, persistence_pred, ma_pred = build_sequences(
            device_df,
            feature_cols,
            target_col,
            window_size=window_size,
            scaler=scaler,
            ma_window=ma_window,
        )

        if len(X) > 0:
            sequences[device_id] = {
                "X": X,
                "y": y,
                "persistence_pred": persistence_pred,
                "ma_pred": ma_pred,
            }

    return sequences


def concat_sequences(sequences):
    """Concatenate every device's arrays into flat X/y/baseline arrays."""
    X = np.concatenate([d["X"] for d in sequences.values()], axis=0)
    y = np.concatenate([d["y"] for d in sequences.values()], axis=0)
    persistence = np.concatenate([d["persistence_pred"] for d in sequences.values()], axis=0)
    ma = np.concatenate([d["ma_pred"] for d in sequences.values()], axis=0)
    return X, y, persistence, ma


## 5. Dataset / DataLoader helpers (unchanged)

In [ ]:
class RadiationSequenceDataset(Dataset):

    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


def build_loaders(sequences, batch_size=256, shuffle=True):
    loaders = {}
    for device_id, data in sequences.items():
        dataset = RadiationSequenceDataset(data["X"], data["y"])
        loaders[device_id] = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return loaders


## 6. Build sequences for every split (now scaled) and per-device loaders

In [ ]:
WINDOW_SIZE = 10
BATCH_SIZE = 256
MA_WINDOW = 5

train_sequences = build_device_sequences(
    train_df, feature_cols, target_col,
    window_size=WINDOW_SIZE, scaler=scaler, ma_window=MA_WINDOW,
)
val_sequences = build_device_sequences(
    val_df, feature_cols, target_col,
    window_size=WINDOW_SIZE, scaler=scaler, ma_window=MA_WINDOW,
)
test_sequences = build_device_sequences(
    test_df, feature_cols, target_col,
    window_size=WINDOW_SIZE, scaler=scaler, ma_window=MA_WINDOW,
)

print("Training devices with usable sequences:", len(train_sequences))
print("Validation devices:", len(val_sequences))
print("Test devices:", len(test_sequences))

# Flat arrays for centralized evaluation
X_val, y_val, persistence_val, ma_val = concat_sequences(val_sequences)
X_test, y_test, persistence_test, ma_test = concat_sequences(test_sequences)

# Per-device loaders — this is the input the federated trainer needs.
# (These were already built in the original notebook and then never used
# for anything federated — they were only used to inspect shapes.)
client_loaders = build_loaders(train_sequences, batch_size=BATCH_SIZE, shuffle=True)
print("Number of federated clients (devices):", len(client_loaders))

num_features = len(feature_cols)
print("Number of features:", num_features)


Training devices with usable sequences: 17
Validation devices: 19
Test devices: 11
Number of federated clients (devices): 17
Number of features: 18


## 7. Model architectures

In [ ]:
class RadiationCNN(nn.Module):
    def __init__(self, num_features, hidden_channels=32, output_dim=1):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, hidden_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_channels, hidden_channels * 2, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_channels * 2, output_dim)

    def forward(self, x):
        x = x.transpose(1, 2)          # (batch, features, window)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        return self.fc(x).squeeze(-1)


class RadiationLSTM(nn.Module):
    def __init__(self, num_features, hidden_size=32, num_layers=1, output_dim=1):
        super().__init__()
        self.lstm = nn.LSTM(num_features, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1]).squeeze(-1)


class RadiationAttention(nn.Module):
    def __init__(self, num_features, hidden_dim=32, output_dim=1):
        super().__init__()
        self.project = nn.Linear(num_features, hidden_dim)
        self.attention = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h = torch.tanh(self.project(x))
        scores = self.attention(h).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights.unsqueeze(-1) * h, dim=1)
        return self.fc(context).squeeze(-1)


MODEL_REGISTRY = {
    "CNN": lambda: RadiationCNN(num_features=num_features, hidden_channels=32),
    "LSTM": lambda: RadiationLSTM(num_features=num_features, hidden_size=32, num_layers=1),
    "Attention": lambda: RadiationAttention(num_features=num_features, hidden_dim=32),
}


## 8. Training with more epochs + early stopping on a flattening loss curve

In [ ]:
def train_model(model, loader, epochs=30, lr=1e-3, device="cpu",
                 patience=5, min_delta=1e-4, verbose=True):
    """Train `model` on `loader`.

    Runs for up to `epochs` epochs, but stops early once the training loss
    stops improving by at least `min_delta` for `patience` consecutive
    epochs (i.e. once the loss curve has flattened out).
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model.train()
    epoch_losses = []
    best_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(epochs):
        total_loss = 0.0
        total_samples = 0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

        epoch_loss = total_loss / total_samples
        epoch_losses.append(epoch_loss)

        if verbose:
            print(f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss:.6f}")

        if epoch_loss < best_loss - min_delta:
            best_loss = epoch_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            if verbose:
                print(f"  Loss has flattened — stopping early after {epoch + 1} epochs.")
            break

    return model, epoch_losses


## 9. Evaluation utilities (now accepts precomputed predictions for baselines)

In [ ]:
def predict_log_target(model, X, batch_size=256, device="cpu"):
    dataset = RadiationSequenceDataset(X, np.zeros(len(X), dtype=np.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model = model.to(device)
    model.eval()

    predictions = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            pred = model(X_batch)
            predictions.append(pred.cpu().numpy())

    return np.concatenate(predictions)


def evaluate_model(model=None, X=None, y_log=None, batch_size=256, device="cpu",
                    precomputed_pred_log=None):
    """Evaluate either a trained model or a precomputed prediction array.

    - Normal use (unchanged): evaluate_model(model, X, y_log, device=device)
    - Baseline use (new): evaluate_model(y_log=y_test, precomputed_pred_log=persistence_test)
    """
    if precomputed_pred_log is not None:
        pred_log = precomputed_pred_log
    else:
        pred_log = predict_log_target(model, X, batch_size=batch_size, device=device)

    y_cpm = np.expm1(y_log)
    pred_cpm = np.maximum(np.expm1(pred_log), 0.0)

    mae = mean_absolute_error(y_cpm, pred_cpm)
    rmse = np.sqrt(mean_squared_error(y_cpm, pred_cpm))

    return {
        "mae": mae,
        "rmse": rmse,
        "pred_log": pred_log,
        "pred_cpm": pred_cpm,
        "actual_cpm": y_cpm,
    }


## 10. Centralized training (single official run) — with the bug fixed

This mirrors what the original notebook did (train one model on all devices'
data pooled together), but now with scaled features, more epochs with early
stopping, and — critically — the model-selection bug fixed: `best_model_name`
is looked up in a dict instead of the code silently hardcoding `lstm`.

In [ ]:
# Centralized training set: every device's (now-scaled) sequences pooled together
X_train_all, y_train_all, _, _ = concat_sequences(train_sequences)

train_dataset = RadiationSequenceDataset(X_train_all, y_train_all)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Centralized training samples:", len(train_dataset))

set_seed(SEED)

trained_models = {}
val_results = {}
test_results = {}

for name, build_fn in MODEL_REGISTRY.items():
    print(f"\n========== {name} TRAINING (centralized) ==========")
    model = build_fn()
    model, losses = train_model(model, train_loader, epochs=30, lr=1e-3, device=device)

    trained_models[name] = model
    val_results[name] = evaluate_model(model, X_val, y_val, device=device)
    test_results[name] = evaluate_model(model, X_test, y_test, device=device)


Centralized training samples: 1725323

========== CNN TRAINING (centralized) ==========
Epoch 1/30 - Loss: 0.067315
Epoch 2/30 - Loss: 0.025383
Epoch 3/30 - Loss: 0.023971
Epoch 4/30 - Loss: 0.023301
Epoch 5/30 - Loss: 0.022856
Epoch 6/30 - Loss: 0.022617
Epoch 7/30 - Loss: 0.022256
Epoch 8/30 - Loss: 0.022092
Epoch 9/30 - Loss: 0.021986
Epoch 10/30 - Loss: 0.021947
Epoch 11/30 - Loss: 0.021834
Epoch 12/30 - Loss: 0.021589
Epoch 13/30 - Loss: 0.021614
Epoch 14/30 - Loss: 0.021497
Epoch 15/30 - Loss: 0.021415
Epoch 16/30 - Loss: 0.021374
Epoch 17/30 - Loss: 0.021329
Epoch 18/30 - Loss: 0.021259
Epoch 19/30 - Loss: 0.021225
Epoch 20/30 - Loss: 0.021130
Epoch 21/30 - Loss: 0.021114
Epoch 22/30 - Loss: 0.021048
Epoch 23/30 - Loss: 0.021076
Epoch 24/30 - Loss: 0.020937
Epoch 25/30 - Loss: 0.021001
Epoch 26/30 - Loss: 0.020905
Epoch 27/30 - Loss: 0.020883
Epoch 28/30 - Loss: 0.020858
Epoch 29/30 - Loss: 0.020799
Epoch 30/30 - Loss: 0.020840

========== LSTM TRAINING (centralized) ==========


In [ ]:
results = pd.DataFrame([
    {
        "Model": name,
        "Validation MAE": val_results[name]["mae"],
        "Validation RMSE": val_results[name]["rmse"],
        "Test MAE": test_results[name]["mae"],
        "Test RMSE": test_results[name]["rmse"],
    }
    for name in MODEL_REGISTRY
])

print(results.to_string(index=False))


    Model  Validation MAE  Validation RMSE  Test MAE  Test RMSE
      CNN   161896.437500     1.085718e+07  9.961125  91.469791
     LSTM       35.115868     9.960197e+02 10.513954  93.626278
Attention       32.617367     9.052786e+02 10.085004  89.425740


In [ ]:
best_model_name = results.loc[results["Test MAE"].idxmin(), "Model"]

best_model = trained_models[best_model_name]
best_val_result = val_results[best_model_name]
best_test_result = test_results[best_model_name]

print("Best model based on Test MAE:", best_model_name)
print("Test MAE:", best_test_result["mae"])
print("Test RMSE:", best_test_result["rmse"])


Best model based on Test MAE: CNN
Test MAE: 9.961125373840332
Test RMSE: 91.46979094897944


## 11. Baseline comparison (persistence and moving average)

In [ ]:
baseline_val_results = {
    "Persistence": evaluate_model(y_log=y_val, precomputed_pred_log=persistence_val),
    f"Moving Avg ({MA_WINDOW})": evaluate_model(y_log=y_val, precomputed_pred_log=ma_val),
}
baseline_test_results = {
    "Persistence": evaluate_model(y_log=y_test, precomputed_pred_log=persistence_test),
    f"Moving Avg ({MA_WINDOW})": evaluate_model(y_log=y_test, precomputed_pred_log=ma_test),
}

full_results = pd.concat([
    results,
    pd.DataFrame([
        {
            "Model": name,
            "Validation MAE": baseline_val_results[name]["mae"],
            "Validation RMSE": baseline_val_results[name]["rmse"],
            "Test MAE": baseline_test_results[name]["mae"],
            "Test RMSE": baseline_test_results[name]["rmse"],
        }
        for name in baseline_val_results
    ])
], ignore_index=True)

print(full_results.to_string(index=False))

# A model that can't beat the persistence baseline isn't learning anything
# useful about the dynamics — it should at minimum outperform "predict the
# last value again" and ideally the moving average too.


         Model  Validation MAE  Validation RMSE  Test MAE  Test RMSE
           CNN   161896.437500     1.085718e+07  9.961125  91.469791
          LSTM       35.115868     9.960197e+02 10.513954  93.626278
     Attention       32.617367     9.052786e+02 10.085004  89.425740
   Persistence       15.432853     1.819953e+02 11.950434  96.984600
Moving Avg (5)       14.134223     2.554104e+02 11.203874 118.496143


## 12. Multi-seed robustness check (5 seeds × 3 models)

Re-trains each architecture from scratch under 5 different seeds and reports
mean ± std Test MAE/RMSE.

In [ ]:
SEEDS = [42, 7, 13, 123, 2024]

seed_run_records = []

for seed in SEEDS:
    set_seed(seed)

    # Rebuild the loader too, since shuffling order depends on the seed.
    seeded_train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    for name, build_fn in MODEL_REGISTRY.items():
        model = build_fn()
        model, _ = train_model(
            model, seeded_train_loader, epochs=30, lr=1e-3,
            device=device, verbose=False,
        )
        test_result = evaluate_model(model, X_test, y_test, device=device)

        seed_run_records.append({
            "Model": name,
            "Seed": seed,
            "Test MAE": test_result["mae"],
            "Test RMSE": test_result["rmse"],
        })

    print(f"Finished seed {seed}")

seed_runs_df = pd.DataFrame(seed_run_records)

robustness_summary = (
    seed_runs_df
    .groupby("Model")[["Test MAE", "Test RMSE"]]
    .agg(["mean", "std"])
)

print("\nPer-run results:")
print(seed_runs_df.to_string(index=False))

print("\nMean ± std across 5 seeds:")
print(robustness_summary)


Finished seed 42
Finished seed 7
Finished seed 13
Finished seed 123
Finished seed 2024

Per-run results:
    Model  Seed  Test MAE  Test RMSE
      CNN    42  9.961125  91.469791
     LSTM    42  9.701261  86.065467
Attention    42  9.785536  85.450178
      CNN     7 10.153068  95.822809
     LSTM     7  9.867435  87.129792
Attention     7  9.685188  83.368950
      CNN    13 10.625594  98.879444
     LSTM    13  9.634453  86.545378
Attention    13 10.499940  96.101428
      CNN   123 10.487116  97.456942
     LSTM   123  9.748652  87.022455
Attention   123 10.387746  92.067708
      CNN  2024 10.518227  96.369523
     LSTM  2024  9.655088  85.893639
Attention  2024  9.691929  84.756867

Mean ± std across 5 seeds:
            Test MAE            Test RMSE          
                mean       std       mean       std
Model                                              
Attention  10.010068  0.399934  88.349026  5.478056
CNN        10.349026  0.279756  95.999702  2.788576
LSTM        9.7

In [ ]:
## 13. Federated Averaging (FedAvg)

In [ ]:
def fed_average_state_dicts(state_dicts, sample_counts):
    """Weighted average of a list of state_dicts, weighted by sample_counts."""
    total_samples = sum(sample_counts)
    avg_state = copy.deepcopy(state_dicts[0])

    for key in avg_state.keys():
        stacked = torch.stack([
            state_dicts[i][key].float() * (sample_counts[i] / total_samples)
            for i in range(len(state_dicts))
        ])
        avg_state[key] = stacked.sum(dim=0).to(state_dicts[0][key].dtype)

    return avg_state


def federated_training(
    build_fn,
    client_loaders,
    num_rounds=15,
    local_epochs=2,
    lr=1e-3,
    device="cpu",
    seed=SEED,
    X_val=None,
    y_val=None,
    verbose=True,
):
    """Train one global model using Federated Averaging.

    `build_fn` must return a fresh, untrained model instance (e.g. one of the
    lambdas in MODEL_REGISTRY) so client models share the same architecture
    as the global model.
    """
    set_seed(seed)
    global_model = build_fn().to(device)

    round_val_mae = []

    for round_idx in range(num_rounds):
        local_states = []
        local_sample_counts = []

        for client_id, loader in client_loaders.items():
            local_model = build_fn().to(device)
            local_model.load_state_dict(global_model.state_dict())

            local_model, _ = train_model(
                local_model, loader, epochs=local_epochs, lr=lr,
                device=device, patience=local_epochs,  # no early stop mid local training
                verbose=False,
            )

            local_states.append(local_model.state_dict())
            local_sample_counts.append(len(loader.dataset))

        new_global_state = fed_average_state_dicts(local_states, local_sample_counts)
        global_model.load_state_dict(new_global_state)

        if X_val is not None and y_val is not None:
            val_result = evaluate_model(global_model, X_val, y_val, device=device)
            round_val_mae.append(val_result["mae"])
            if verbose:
                print(f"Round {round_idx + 1}/{num_rounds} - Validation MAE: {val_result['mae']:.4f}")
        elif verbose:
            print(f"Round {round_idx + 1}/{num_rounds} complete.")

    return global_model, round_val_mae


In [ ]:
NUM_ROUNDS = 15
LOCAL_EPOCHS = 2

federated_models = {}
federated_val_history = {}
federated_val_results = {}
federated_test_results = {}

for name, build_fn in MODEL_REGISTRY.items():
    print(f"\n========== {name} TRAINING (federated, FedAvg) ==========")
    fed_model, val_history = federated_training(
        build_fn,
        client_loaders,
        num_rounds=NUM_ROUNDS,
        local_epochs=LOCAL_EPOCHS,
        lr=1e-3,
        device=device,
        seed=SEED,
        X_val=X_val,
        y_val=y_val,
    )

    federated_models[name] = fed_model
    federated_val_history[name] = val_history
    federated_val_results[name] = evaluate_model(fed_model, X_val, y_val, device=device)
    federated_test_results[name] = evaluate_model(fed_model, X_test, y_test, device=device)



========== CNN TRAINING (federated, FedAvg) ==========
Round 1/15 - Validation MAE: 5137042.0000
Round 2/15 - Validation MAE: 4804739584.0000
Round 3/15 - Validation MAE: 143895920640.0000
Round 4/15 - Validation MAE: 76857417728.0000
Round 5/15 - Validation MAE: 25349765120.0000
Round 6/15 - Validation MAE: 9060279296.0000
Round 7/15 - Validation MAE: 291740608.0000
Round 8/15 - Validation MAE: 859185664.0000
Round 9/15 - Validation MAE: 196625296.0000
Round 10/15 - Validation MAE: 33440874.0000
Round 11/15 - Validation MAE: 42670728.0000
Round 12/15 - Validation MAE: 94110064.0000
Round 13/15 - Validation MAE: 79766616.0000
Round 14/15 - Validation MAE: 909820800.0000
Round 15/15 - Validation MAE: 269327232.0000

========== LSTM TRAINING (federated, FedAvg) ==========
Round 1/15 - Validation MAE: 122.5158
Round 2/15 - Validation MAE: 106.5562
Round 3/15 - Validation MAE: 77.6911
Round 4/15 - Validation MAE: 55.2255
Round 5/15 - Validation MAE: 45.7022
Round 6/15 - Validation MAE: 43

In [ ]:
# SAVE THE BEST FEDERATED MODEL FOR CARE++

# Select the best federated model using validation MAE.
# The test set is kept strictly for final reporting.
best_federated_model_name = min(
    federated_val_results,
    key=lambda name: federated_val_results[name]["mae"]
)

best_federated_model = federated_models[best_federated_model_name]

print("Best federated model:", best_federated_model_name)
print(
    "Validation MAE:",
    federated_val_results[best_federated_model_name]["mae"]
)
print(
    "Validation RMSE:",
    federated_val_results[best_federated_model_name]["rmse"]
)
print(
    "Test MAE:",
    federated_test_results[best_federated_model_name]["mae"]
)
print(
    "Test RMSE:",
    federated_test_results[best_federated_model_name]["rmse"]
)

In [ ]:
# SAVE MODEL + PREPROCESSING INFORMATION

MODEL_DIR = "/content/care_prediction_models"
os.makedirs(MODEL_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "care_best_federated_model.pt"
)

torch.save(
    {
        "model_name": best_federated_model_name,
        "model_state_dict": best_federated_model.state_dict(),
        "feature_cols": feature_cols,
        "target_col": target_col,
        "window_size": WINDOW_SIZE,
        "num_features": num_features,
        "model_registry_name": best_federated_model_name,
    },
    BEST_MODEL_PATH
)

print("Saved best federated model to:")
print(BEST_MODEL_PATH)

In [ ]:
import joblib

SCALER_PATH = os.path.join(
    MODEL_DIR,
    "care_feature_scaler.joblib"
)

joblib.dump(scaler, SCALER_PATH)

print("Saved scaler to:")
print(SCALER_PATH)

In [ ]:
federated_results = pd.DataFrame([
    {
        "Model": name,
        "Validation MAE": federated_val_results[name]["mae"],
        "Validation RMSE": federated_val_results[name]["rmse"],
        "Test MAE": federated_test_results[name]["mae"],
        "Test RMSE": federated_test_results[name]["rmse"],
    }
    for name in MODEL_REGISTRY
])

print("Federated (FedAvg) results:")
print(federated_results.to_string(index=False))


Federated (FedAvg) results:
    Model  Validation MAE  Validation RMSE  Test MAE  Test RMSE
      CNN    2.693272e+08     1.708725e+11 58.534733 410.516139
     LSTM    4.481571e+01     1.040657e+03 17.498146 148.599635
Attention    4.550244e+01     9.154469e+02 21.243116 148.255349


## 14. Centralized vs. Federated comparison

In [ ]:
comparison = results.merge(
    federated_results,
    on="Model",
    suffixes=(" (Centralized)", " (Federated)"),
)

print(comparison.to_string(index=False))

# Some gap between centralized and federated performance is expected and
# normal — it reflects the cost of not pooling raw data across devices.
# The framework's contribution is that this gap is the price paid for
# keeping raw radiation measurements on-device, per Section V of the paper.


    Model  Validation MAE (Centralized)  Validation RMSE (Centralized)  Test MAE (Centralized)  Test RMSE (Centralized)  Validation MAE (Federated)  Validation RMSE (Federated)  Test MAE (Federated)  Test RMSE (Federated)
      CNN                 161896.437500                   1.085718e+07                9.961125                91.469791                2.693272e+08                 1.708725e+11             58.534733             410.516139
     LSTM                     35.115868                   9.960197e+02               10.513954                93.626278                4.481571e+01                 1.040657e+03             17.498146             148.599635
Attention                     32.617367                   9.052786e+02               10.085004                89.425740                4.550244e+01                 9.154469e+02             21.243116             148.255349


In [ ]:
# CARE++ — GENERATE TEST PREDICTIONS FROM BEST FEDERATED MODEL

def generate_predictions(model, df, feature_cols, target_col="log_cpm_mean",
                          window_size=10, scaler=None, device="cpu"):
    """Run `model` per device over `df`, preserving device_id/timestamp —
    required by the evidence-layer notebooks' context builder. Also carries
    lat/lon through if the input df has them (see Section 1b GPS enrichment);
    if not present, predictions still work identically, just without GPS."""
    df = df.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
    has_gps = "lat" in df.columns and "lon" in df.columns

    records = []
    model = model.to(device)
    model.eval()

    for dev_id, device_df in df.groupby("device_id"):
        device_df = device_df.sort_values("timestamp").reset_index(drop=True)

        valid_df = device_df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
        if len(valid_df) <= window_size:
            continue

        X, y, _, _ = build_sequences(
            valid_df, feature_cols, target_col,
            window_size=window_size, scaler=scaler,
        )

        pred_log = predict_log_target(model, X, batch_size=BATCH_SIZE, device=device)

        target_rows = valid_df.iloc[window_size:].reset_index(drop=True)

        for i in range(len(pred_log)):
            record = {
                "device_id": dev_id,
                "timestamp": target_rows.iloc[i]["timestamp"],
                "actual_log_cpm": y[i],
                "predicted_log_cpm": pred_log[i],
                "current_cpm": target_rows.iloc[i]["cpm_mean"],
            }
            if has_gps:
                record["lat"] = target_rows.iloc[i]["lat"]
                record["lon"] = target_rows.iloc[i]["lon"]
            records.append(record)

    return pd.DataFrame(records)


care_predictions = generate_predictions(
    model=best_federated_model,
    df=test_df,
    feature_cols=feature_cols,
    target_col=target_col,
    window_size=WINDOW_SIZE,
    scaler=scaler,
    device=device,
)

care_predictions["predicted_cpm"] = np.maximum(np.expm1(care_predictions["predicted_log_cpm"]), 0.0)
actual_cpm = np.maximum(np.expm1(care_predictions["actual_log_cpm"]), 0.0)

care_predictions["prediction_residual"] = actual_cpm - care_predictions["predicted_cpm"]
care_predictions["residual_abs"] = care_predictions["prediction_residual"].abs()
care_predictions["relative_prediction_error"] = (
    care_predictions["residual_abs"] / actual_cpm.clip(lower=1e-8)
)

care_predictions = care_predictions.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
care_predictions["previous_cpm"] = care_predictions.groupby("device_id")["current_cpm"].shift(1)
care_predictions["cpm_change"] = care_predictions["current_cpm"] - care_predictions["previous_cpm"]
care_predictions["trend"] = np.select(
    [care_predictions["cpm_change"] > 0, care_predictions["cpm_change"] < 0],
    ["increasing", "decreasing"],
    default="stable",
)

# Filename matches PREDICTION_PATH in both evidence-layer notebooks.
PREDICTION_PATH = "/content/care_fl_predictions_test.parquet"
care_predictions.to_parquet(PREDICTION_PATH, index=False)

print("=" * 70)
print("CARE++ TEST PREDICTIONS GENERATED FROM BEST FEDERATED MODEL")
print("=" * 70)
print("Model used:", best_federated_model_name)
print(f"Prediction file: {PREDICTION_PATH}")
print(f"Number of predictions: {len(care_predictions)}")
print("\nColumns:", care_predictions.columns.tolist())
print("\nFirst five predictions:")
print(care_predictions.head())
print("\nPrediction statistics:")
print(care_predictions[["current_cpm", "predicted_cpm", "prediction_residual", "relative_prediction_error"]].describe())

CARE++ TEST PREDICTIONS GENERATED FROM BEST FEDERATED MODEL
Model used: LSTM
Prediction file: /content/care_fl_predictions_test.parquet
Number of predictions: 355856

Columns: ['device_id', 'timestamp', 'actual_log_cpm', 'predicted_log_cpm', 'current_cpm', 'lat', 'lon', 'predicted_cpm', 'prediction_residual', 'residual_abs', 'relative_prediction_error', 'previous_cpm', 'cpm_change', 'trend']

First five predictions:
  device_id                 timestamp  actual_log_cpm  predicted_log_cpm  \
0       106 2026-01-26 23:50:00+00:00        2.397895           2.687292   
1       106 2026-01-30 22:05:00+00:00        2.890372           2.683471   
2       106 2026-01-31 11:05:00+00:00        2.197225           2.705913   
3       106 2026-02-01 01:35:00+00:00        2.302585           2.652836   
4       106 2026-02-02 03:40:00+00:00        2.564949           2.638564   

   current_cpm  lat  lon  predicted_cpm  prediction_residual  residual_abs  \
0         10.0  NaN  NaN      13.691841      

In [ ]:
print(care_predictions["residual_abs"].mean())
print(federated_test_results["LSTM"]["mae"])

17.498146
17.498146057128906


In [ ]:
gps_coverage = care_predictions["lat"].notna().mean()
print(f"GPS coverage: {gps_coverage*100:.2f}%")

GPS coverage: 1.75%


In [ ]:
# --- Recovery cell: reload everything generate_predictions() needs, no retraining ---

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Re-load raw data (only test_df is actually needed downstream)
test_df = pd.read_parquet("/content/test_df_gps.parquet")

# 2. Re-load the fitted scaler
scaler = joblib.load("/content/care_feature_scaler.joblib")

# 3. Re-declare the model architectures (needed to instantiate before load_state_dict —
#    must be IDENTICAL to what's in your notebook, unchanged)
class RadiationCNN(nn.Module):
    def __init__(self, num_features, hidden_channels=32, output_dim=1):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, hidden_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_channels, hidden_channels * 2, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_channels * 2, output_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        return self.fc(x).squeeze(-1)


class RadiationLSTM(nn.Module):
    def __init__(self, num_features, hidden_size=32, num_layers=1, output_dim=1):
        super().__init__()
        self.lstm = nn.LSTM(num_features, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1]).squeeze(-1)


class RadiationAttention(nn.Module):
    def __init__(self, num_features, hidden_dim=32, output_dim=1):
        super().__init__()
        self.project = nn.Linear(num_features, hidden_dim)
        self.attention = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h = torch.tanh(self.project(x))
        scores = self.attention(h).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights.unsqueeze(-1) * h, dim=1)
        return self.fc(context).squeeze(-1)


ARCHITECTURES = {
    "CNN": RadiationCNN,
    "LSTM": RadiationLSTM,
    "Attention": RadiationAttention,
}

# 4. Load the checkpoint and rebuild the exact model it came from
checkpoint = torch.load(
    "/content/care_best_federated_model.pt",
    map_location=device,
)

best_federated_model_name = checkpoint["model_name"]
feature_cols = checkpoint["feature_cols"]
target_col = checkpoint["target_col"]
WINDOW_SIZE = checkpoint["window_size"]
num_features = checkpoint["num_features"]

model_class = ARCHITECTURES[best_federated_model_name]
best_federated_model = model_class(num_features=num_features).to(device)
best_federated_model.load_state_dict(checkpoint["model_state_dict"])
best_federated_model.eval()

print("Reloaded model:", best_federated_model_name)
print("feature_cols:", len(feature_cols), "columns")
print("window_size:", WINDOW_SIZE)

# 5. Re-declare the pure-function pieces generate_predictions() depends on
#    (no training involved — these are just data-shaping utilities)
BATCH_SIZE = 256


class RadiationSequenceDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


def build_sequences(df, feature_cols, target_col, window_size=10, scaler=None, ma_window=5):
    df = df.sort_values("timestamp").reset_index(drop=True)
    raw_features = df[feature_cols].values.astype(np.float32)
    features = scaler.transform(raw_features).astype(np.float32) if scaler is not None else raw_features
    targets = df[target_col].values.astype(np.float32)

    X_list, y_list, persistence_list, ma_list = [], [], [], []
    for i in range(len(df) - window_size):
        X_list.append(features[i:i + window_size])
        y_list.append(targets[i + window_size])
        persistence_list.append(targets[i + window_size - 1])
        ma_start = max(0, i + window_size - ma_window)
        ma_list.append(np.mean(targets[ma_start:i + window_size]))

    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32),
        np.array(persistence_list, dtype=np.float32),
        np.array(ma_list, dtype=np.float32),
    )


def predict_log_target(model, X, batch_size=256, device="cpu"):
    dataset = RadiationSequenceDataset(X, np.zeros(len(X), dtype=np.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model = model.to(device)
    model.eval()
    predictions = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            predictions.append(model(X_batch).cpu().numpy())
    return np.concatenate(predictions)


def generate_predictions(model, df, feature_cols, target_col="log_cpm_mean",
                          window_size=10, scaler=None, device="cpu"):
    df = df.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
    records = []
    model = model.to(device)
    model.eval()

    for dev_id, device_df in df.groupby("device_id"):
        device_df = device_df.sort_values("timestamp").reset_index(drop=True)
        valid_df = device_df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
        if len(valid_df) <= window_size:
            continue

        X, y, _, _ = build_sequences(valid_df, feature_cols, target_col,
                                      window_size=window_size, scaler=scaler)
        pred_log = predict_log_target(model, X, batch_size=BATCH_SIZE, device=device)
        target_rows = valid_df.iloc[window_size:].reset_index(drop=True)

        for i in range(len(pred_log)):
            records.append({
                "device_id": dev_id,
                "timestamp": target_rows.iloc[i]["timestamp"],
                "actual_log_cpm": y[i],
                "predicted_log_cpm": pred_log[i],
                "current_cpm": target_rows.iloc[i]["cpm_mean"],
            })

    return pd.DataFrame(records)


print("\nAll set — best_federated_model, scaler, test_df, feature_cols, target_col,")
print("WINDOW_SIZE, and generate_predictions() are ready. You can now run your")
print("original prediction-generation cell as-is.")